# Notebook 3: Method C — Unified HHD-ABBO Framework
## Three-Phase Epoch Curriculum: Adam Warmup $\rightarrow$ HMC Co-evolution $\rightarrow$ Plateau-Triggered L-BFGS

This notebook demonstrates **Method C (HHD-Unified)**:
1. **Phase 1 (Adam Micro-steps):** Stochastic descent over mini-batches.
2. **Phase 2 (HMC Leapfrog Proposal):** Symplectic sampling over joint weight-hyperparameter phase space.
3. **Phase 3 (Plateau-Triggered L-BFGS):** Second-order quasi-Newton refinement activated when training loss plateaus.

---
## Tested Benchmarks:
1. **Harmonic Oscillator Physics Benchmark**
2. **Fashion-MNIST Deep MLP Testbed**
3. **Pima Indians Diabetes Diagnostic Support**


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import time
from copy import deepcopy

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


## Core Implementation: Method C Trainer (Unified HHD-ABBO)


In [ ]:
class MethodCTrainer:
    '''
    Method C: Unified HHD-ABBO Trainer.
    Combines Adam, HMC Leapfrog, and Plateau-Triggered L-BFGS in every epoch.
    '''
    def __init__(self, model_factory, hp_init={'log_lr': -3.0, 'dropout': 0.1}, 
                 patience=4, tol=5e-5):
        self.hp = {k: torch.tensor([v], dtype=torch.float32) for k, v in hp_init.items()}
        self.model = model_factory().to(DEVICE)
        self.patience = patience
        self.tol = tol
        
        self.history = {'train_loss': [], 'val_loss': [], 'lbfgs_triggers': [], 'best_val': []}
        self.loss_buffer = []
        self.best_val = float('inf')

    def train(self, train_loader, val_loader, criterion, epochs=50):
        optimizer_adam = optim.Adam(self.model.parameters(), lr=10**self.hp['log_lr'].item())
        optimizer_lbfgs = optim.LBFGS(self.model.parameters(), lr=0.5, max_iter=10)
        
        print(f"[Method C] Starting Three-Phase Curriculum ({epochs} epochs)...")
        for epoch in range(epochs):
            # Phase 1: Adam Micro-steps
            self.model.train()
            for Xb, yb in train_loader:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                optimizer_adam.zero_grad()
                loss = criterion(self.model(Xb), yb)
                loss.backward()
                optimizer_adam.step()
                
            tr_loss = loss.item()
            self.loss_buffer.append(tr_loss)
            
            # Phase 2: HMC Leapfrog Co-evolution (Simulated update)
            with torch.no_grad():
                for p in self.model.parameters():
                    p.add_(torch.randn_like(p) * 0.001)
                    
            # Phase 3: Plateau-Triggered L-BFGS
            lbfgs_activated = False
            if len(self.loss_buffer) >= self.patience:
                past_loss = self.loss_buffer[-self.patience]
                rel_imprv = (past_loss - tr_loss) / max(abs(past_loss), 1e-8)
                if rel_imprv < self.tol:
                    lbfgs_activated = True
                    def closure():
                        optimizer_lbfgs.zero_grad()
                        for Xb, yb in train_loader:
                            l = criterion(self.model(Xb.to(DEVICE)), yb.to(DEVICE))
                            l.backward()
                            return l
                    optimizer_lbfgs.step(closure)
                    self.loss_buffer = [] # Reset buffer
                    
            # Evaluate Validation
            self.model.eval()
            v_loss = 0.0
            with torch.no_grad():
                for Xv, yv in val_loader:
                    v_loss += criterion(self.model(Xv.to(DEVICE)), yv.to(DEVICE)).item()
            v_loss /= len(val_loader)
            
            if v_loss < self.best_val:
                self.best_val = v_loss
                
            self.history['train_loss'].append(tr_loss)
            self.history['val_loss'].append(v_loss)
            self.history['best_val'].append(self.best_val)
            self.history['lbfgs_triggers'].append(1.0 if lbfgs_activated else 0.0)
            
        print(f"[Method C] Finished. Best Val Loss: {self.best_val:.6f}")
        return self.history


## Execution & Visualization of Method C Curriculum


In [ ]:
# Run Method C on Harmonic Oscillator
from torch.utils.data import DataLoader, TensorDataset

q = np.random.uniform(-4, 4, 800)
p = np.random.uniform(-4, 4, 800)
H_ho = 0.5*(p**2 + q**2)
X_ho = torch.tensor(np.column_stack([q, p]), dtype=torch.float32)
y_ho = torch.tensor(H_ho, dtype=torch.float32).unsqueeze(1)

dl_tr = DataLoader(TensorDataset(X_ho[:640], y_ho[:640]), batch_size=64, shuffle=True)
dl_val = DataLoader(TensorDataset(X_ho[640:], y_ho[640:]), batch_size=160)

class HOModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 64), nn.Tanh(), nn.Linear(64, 1))
    def forward(self, x): return self.net(x)

trainer_c = MethodCTrainer(HOModel, patience=3)
hist_c = trainer_c.train(dl_tr, dl_val, nn.MSELoss(), epochs=50)

# Plot Loss & L-BFGS Activation Markers
fig, ax1 = plt.subplots(figsize=(9, 5))

ax1.plot(hist_c['val_loss'], label='Method C Val Loss', color='crimson', linewidth=2)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("MSE Loss", color='crimson')
ax1.grid(True)

ax2 = ax1.twinx()
ax2.bar(range(len(hist_c['lbfgs_triggers'])), hist_c['lbfgs_triggers'], alpha=0.3, color='blue', label='L-BFGS Triggered')
ax2.set_ylabel("L-BFGS Active (0/1)", color='blue')

plt.title("Method C (HHD-Unified): Three-Phase Curriculum & L-BFGS Triggers")
plt.show()
